# Notebook 07 — Integración Hardware ↔ API
**Comunicación ESP32/Arduino → FastAPI → Clasificación en tiempo real**

Este notebook cubre:
- Cómo probar la API desde Python (simulando el hardware)
- Cómo conectar el ESP32 real a la API
- Monitoreo continuo y visualización en tiempo real
- Código MicroPython/Arduino para el microcontrolador


## Requisito previo — Levantar el backend FastAPI

Antes de ejecutar este notebook, abre una **terminal nueva** y ejecuta:
```
.venv\Scripts\uvicorn.exe src.main:app --reload --host 0.0.0.0 --port 8000
```
Luego verifica en `http://localhost:8000/docs` que el servidor responde.


In [ ]:
import httpx
import json
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from datetime import datetime
from IPython.display import display, clear_output
import warnings, os

warnings.filterwarnings('ignore')

API_BASE = 'http://localhost:8000'

print('Librerías cargadas.')
print(f'API objetivo: {API_BASE}')

## 7.1 — Verificar que la API está en línea

In [ ]:
try:
    resp = httpx.get(f'{API_BASE}/', timeout=5)
    print('✅  API en línea')
    print(json.dumps(resp.json(), indent=2, ensure_ascii=False))
except httpx.ConnectError:
    print('❌  API no disponible. Levanta el servidor primero:')
    print('    .venv\\Scripts\\uvicorn.exe src.main:app --reload')

In [ ]:
# Verificar modelos cargados
resp = httpx.get(f'{API_BASE}/monitor/models', timeout=5)
print('Modelos disponibles en la API:')
for m in resp.json():
    estado = '✅' if m['disponible'] else '❌'
    print(f'  {estado} {m["nombre"]:15s} → {m["tipo"]}')
    print(f'     Clases: {m["clases"]}')

---
## 7.2 — Formato del JSON request para `/predict/`

### Campos requeridos
| Campo | Tipo | Rango | Descripción |
|---|---|---|---|
| `ph` | float | 0.0 – 14.0 | Valor de pH del sensor SEN0161 |
| `turbidez` | float | ≥ 0.0 | Turbidez en NTU del sensor SEN0189 |
| `temperatura` | float | 0.0 – 100.0 | Temperatura en °C del DS18B20 |

### Campos opcionales
| Campo | Tipo | Default | Descripción |
|---|---|---|---|
| `punto_monitoreo` | str | `"Sensor-Hardware"` | Nombre del punto de monitoreo |
| `timestamp` | datetime ISO | Hora actual | Fecha y hora de la medición |

### Parámetro de query: `?modelo=`
- `random_forest` → usa el modelo Random Forest (más rápido, recomendado para hardware)
- `rna` → usa la Red Neuronal Artificial


## 7.3 — Ejemplos de requests de prueba

In [ ]:
def predecir(ph, turbidez, temperatura, modelo='random_forest', punto='Prueba-Manual'):
    """Envía una lectura a la API y muestra el resultado formateado."""
    payload = {
        'ph':              ph,
        'turbidez':        turbidez,
        'temperatura':     temperatura,
        'punto_monitoreo': punto,
        'timestamp':       datetime.utcnow().isoformat()
    }
    resp = httpx.post(f'{API_BASE}/predict/?modelo={modelo}',
                      json=payload, timeout=10)
    if resp.status_code == 200:
        r = resp.json()
        icono = {'Optima': '🟢', 'Alerta': '🟡', 'Contaminada': '🔴'}.get(r['estado'], '⚪')
        print(f'{icono}  Estado:     {r["estado"]:12s}  (confianza: {r["confianza"]*100:.1f}%)')
        print(f'   Modelo:     {r["modelo_usado"]}')
        print(f'   Alerta:     {r["alerta_activa"]}')
        print(f'   Probs:      {r["probabilidades"]}')
        print(f'   Consejo:    {r["recomendacion"]}')
        return r
    else:
        print(f'Error {resp.status_code}: {resp.text}')
        return None

In [ ]:
print('=== Caso 1: Agua ÓPTIMA ===')
predecir(ph=7.2, turbidez=2.5, temperatura=23.0, modelo='random_forest')

In [ ]:
print('=== Caso 2: Agua en ALERTA (turbidez alta) ===')
predecir(ph=6.8, turbidez=7.5, temperatura=26.0, modelo='random_forest')

In [ ]:
print('=== Caso 3: Agua CONTAMINADA (pH ácido + turbidez muy alta) ===')
predecir(ph=4.2, turbidez=18.0, temperatura=31.0, modelo='random_forest')

In [ ]:
print('=== Comparar ambos modelos con la misma lectura ===')
lectura = dict(ph=6.3, turbidez=6.0, temperatura=27.5)
print('\n── Random Forest ──')
predecir(**lectura, modelo='random_forest')
print('\n── Red Neuronal ──')
predecir(**lectura, modelo='rna')

---
## 7.4 — Predicción por LOTE (múltiples lecturas en un solo request)

Útil cuando el ESP32 almacena lecturas en la tarjeta SD y las envía en bloque.

In [ ]:
# Simular un lote de lecturas como si vinieran del hardware cada 5 minutos
lote_payload = {
    'lecturas': [
        {'ph': 7.1, 'turbidez': 2.1, 'temperatura': 23.0, 'punto_monitoreo': 'Guatapuri-km3'},
        {'ph': 6.9, 'turbidez': 3.8, 'temperatura': 24.5, 'punto_monitoreo': 'Guatapuri-km3'},
        {'ph': 6.4, 'turbidez': 6.2, 'temperatura': 25.1, 'punto_monitoreo': 'Guatapuri-km3'},
        {'ph': 5.8, 'turbidez': 9.5, 'temperatura': 25.8, 'punto_monitoreo': 'Guatapuri-km3'},
        {'ph': 4.9, 'turbidez': 14.2,'temperatura': 27.0, 'punto_monitoreo': 'Guatapuri-km3'},
    ]
}

resp = httpx.post(f'{API_BASE}/predict/batch?modelo=random_forest',
                  json=lote_payload, timeout=15)
resultados = resp.json()

print(f'Resultados del lote ({len(resultados)} lecturas):')
print(f'{"N°":>3} {"pH":>6} {"Turb":>6} {"Temp":>6} │ {"Estado":>12} {"Confianza":>10}')
print('─' * 55)
for i, (entrada, res) in enumerate(zip(lote_payload['lecturas'], resultados), 1):
    icono = {'Optima':'🟢','Alerta':'🟡','Contaminada':'🔴'}.get(res['estado'],'⚪')
    print(f'{i:3} {entrada["ph"]:6.1f} {entrada["turbidez"]:6.1f} '
          f'{entrada["temperatura"]:6.1f} │ {icono} {res["estado"]:10} {res["confianza"]*100:8.1f}%')

---
## 7.5 — Simulación de monitoreo continuo (cada N segundos)

Simula el flujo de datos que el ESP32 enviaría a la API en tiempo real.

In [ ]:
def simular_sensor(n_lecturas=20, intervalo_s=1.5, modelo='random_forest'):
    """
    Simula lecturas continuas del hardware y grafica el estado en tiempo real.
    n_lecturas: cantidad de lecturas a simular
    intervalo_s: segundos entre cada lectura
    """
    np.random.seed(None)  # seed aleatorio cada vez
    historico = []
    colores_estado = {'Optima': '#2ECC71', 'Alerta': '#F39C12', 'Contaminada': '#E74C3C'}

    # Parámetros base que derivan hacia contaminación gradual
    ph_base    = np.linspace(7.2, 5.5, n_lecturas) + np.random.normal(0, 0.15, n_lecturas)
    turb_base  = np.linspace(2.0, 12.0, n_lecturas) + np.random.normal(0, 0.5, n_lecturas)
    temp_base  = np.linspace(23.0, 28.5, n_lecturas) + np.random.normal(0, 0.3, n_lecturas)

    fig, axes = plt.subplots(2, 2, figsize=(14, 8))
    fig.suptitle('Monitoreo en tiempo real — Simulación Hardware → API', fontsize=13, fontweight='bold')

    for i in range(n_lecturas):
        ph   = round(float(ph_base[i]), 2)
        turb = round(max(0.1, float(turb_base[i])), 2)
        temp = round(float(temp_base[i]), 1)

        # Llamada real a la API
        try:
            payload = {'ph': ph, 'turbidez': turb, 'temperatura': temp,
                       'punto_monitoreo': 'Guatapuri-Simulado'}
            resp    = httpx.post(f'{API_BASE}/predict/?modelo={modelo}',
                                 json=payload, timeout=5)
            r       = resp.json()
            estado  = r['estado']
            confianza = r['confianza']
        except Exception as e:
            estado, confianza = 'Error', 0.0

        historico.append({'t': i+1, 'pH': ph, 'Turbidez': turb,
                          'Temperatura': temp, 'Estado': estado,
                          'Confianza': confianza})
        df_hist = pd.DataFrame(historico)

        # Actualizar gráfico
        for ax in axes.flat:
            ax.clear()

        params_plot = [('pH', '#3498DB'), ('Turbidez', '#1ABC9C'), ('Temperatura', '#E67E22')]
        for ax, (param, color) in zip(axes.flat[:3], params_plot):
            ax.plot(df_hist['t'], df_hist[param], 'o-', color=color, linewidth=2)
            ax.set_title(param, fontweight='bold')
            ax.set_xlabel('Lectura #')
            ax.grid(alpha=0.3)

        # Panel de estado
        ax4 = axes[1, 1]
        conteo = df_hist['Estado'].value_counts()
        colores = [colores_estado.get(e, 'gray') for e in conteo.index]
        ax4.bar(conteo.index, conteo.values, color=colores)
        ax4.set_title(f'Estado actual: {estado}', fontweight='bold',
                      color=colores_estado.get(estado, 'black'))
        ax4.set_ylabel('Cantidad')

        plt.tight_layout()
        clear_output(wait=True)
        display(fig)

        icono = {'Optima':'🟢','Alerta':'🟡','Contaminada':'🔴'}.get(estado,'⚠️')
        print(f'Lectura {i+1:2}/{n_lecturas} │ pH={ph:.2f} Turb={turb:.1f} Temp={temp:.1f} │ '
              f'{icono} {estado} ({confianza*100:.0f}%)')
        time.sleep(intervalo_s)

    plt.close()
    print('\nSimulación completada.')
    return pd.DataFrame(historico)

# Ejecutar simulación (20 lecturas, 1.5s entre cada una)
df_sim = simular_sensor(n_lecturas=20, intervalo_s=1.5)

---
## 7.6 — Código para el ESP32 (MicroPython)

Pega este código en el archivo `main.py` del ESP32 usando **Thonny IDE**.

> Reemplaza `WIFI_SSID`, `WIFI_PASSWORD` y `API_HOST` con tus datos reales.

In [ ]:
codigo_micropython = '''
# ============================================================
# main.py para ESP32 — Sistema de Monitoreo Calidad del Agua
# Universidad Popular del Cesar — Ingeniería de Sistemas 2026
# ============================================================
import network
import urequests
import ujson
import time
from machine import Pin, ADC
import onewire, ds18x20

# ─── Configuración WiFi ──────────────────────────────────────
WIFI_SSID     = "TU_RED_WIFI"
WIFI_PASSWORD = "TU_CONTRASEÑA"
API_HOST      = "http://192.168.1.X:8000"  # IP del PC donde corre FastAPI
API_ENDPOINT  = API_HOST + "/predict/?modelo=random_forest"
INTERVALO_S   = 300   # Leer cada 5 minutos

# ─── Pines de sensores ───────────────────────────────────────
PH_PIN      = ADC(Pin(34))   # SEN0161 → GPIO34 (ADC1)
TURB_PIN    = ADC(Pin(35))   # SEN0189 → GPIO35 (ADC1)
TEMP_PIN    = Pin(4)          # DS18B20 → GPIO4 (OneWire)
LED_OPTIMA  = Pin(25, Pin.OUT)
LED_ALERTA  = Pin(26, Pin.OUT)
LED_CONTAM  = Pin(27, Pin.OUT)

# ─── Calibración sensores ────────────────────────────────────
PH_OFFSET     = 0.0    # Ajustar con solución buffer pH 7.0
TURB_VOLTAJE  = 3.3    # Voltaje de referencia

def conectar_wifi():
    wlan = network.WLAN(network.STA_IF)
    wlan.active(True)
    if not wlan.isconnected():
        print("Conectando a WiFi:", WIFI_SSID)
        wlan.connect(WIFI_SSID, WIFI_PASSWORD)
        timeout = 15
        while not wlan.isconnected() and timeout > 0:
            time.sleep(1)
            timeout -= 1
    if wlan.isconnected():
        print("WiFi conectado:", wlan.ifconfig()[0])
        return True
    print("Error WiFi")
    return False

def leer_ph():
    # Promedio de 10 lecturas para reducir ruido
    PH_PIN.atten(ADC.ATTN_11DB)
    lecturas = [PH_PIN.read() for _ in range(10)]
    voltaje  = (sum(lecturas) / 10) * 3.3 / 4095
    ph       = 3.5 * voltaje + PH_OFFSET  # Fórmula empírica SEN0161
    return round(max(0.0, min(14.0, ph)), 2)

def leer_turbidez():
    TURB_PIN.atten(ADC.ATTN_11DB)
    lecturas = [TURB_PIN.read() for _ in range(10)]
    voltaje  = (sum(lecturas) / 10) * 3.3 / 4095
    # Conversión voltaje → NTU (curva SEN0189)
    if voltaje < 2.5:
        ntu = 3000.0
    else:
        ntu = -1120.4 * voltaje**2 + 5742.3 * voltaje - 4352.9
    return round(max(0.0, ntu), 1)

def leer_temperatura():
    ow  = onewire.OneWire(TEMP_PIN)
    ds  = ds18x20.DS18X20(ow)
    rom = ds.scan()
    if rom:
        ds.convert_temp()
        time.sleep_ms(750)
        return round(ds.read_temp(rom[0]), 1)
    return 25.0  # Valor por defecto si el sensor falla

def activar_leds(estado):
    LED_OPTIMA.value(1 if estado == "Optima"      else 0)
    LED_ALERTA.value(1 if estado == "Alerta"      else 0)
    LED_CONTAM.value(1 if estado == "Contaminada" else 0)

def enviar_lectura(ph, turbidez, temperatura):
    payload = ujson.dumps({
        "ph":              ph,
        "turbidez":        turbidez,
        "temperatura":     temperatura,
        "punto_monitoreo": "Guatapuri-ESP32"
    })
    try:
        resp = urequests.post(
            API_ENDPOINT,
            data=payload,
            headers={"Content-Type": "application/json"},
            timeout=10
        )
        if resp.status_code == 200:
            resultado = resp.json()
            resp.close()
            return resultado
        resp.close()
    except Exception as e:
        print("Error API:", e)
    return None

# ─── Loop principal ──────────────────────────────────────────
def main():
    if not conectar_wifi():
        return

    print("Sistema iniciado. Monitoreando cada", INTERVALO_S, "segundos.")
    while True:
        ph    = leer_ph()
        turb  = leer_turbidez()
        temp  = leer_temperatura()

        print(f"pH={ph} | Turbidez={turb} NTU | Temp={temp}°C")

        resultado = enviar_lectura(ph, turb, temp)
        if resultado:
            estado = resultado.get("estado", "Error")
            print(f"Estado: {estado} | Confianza: {resultado.get('confianza',0)*100:.0f}%")
            print(f"Accion: {resultado.get('recomendacion','')}")
            activar_leds(estado)
        else:
            print("Sin respuesta de la API.")
            # En modo offline: clasificación local simple
            if ph < 6.0 or turb > 10.0:
                activar_leds("Contaminada")
            elif ph < 6.5 or turb > 5.0:
                activar_leds("Alerta")
            else:
                activar_leds("Optima")

        time.sleep(INTERVALO_S)

main()
'''

# Guardar el código en firmware/
os.makedirs('../firmware', exist_ok=True)
with open('../firmware/esp32_main.py', 'w', encoding='utf-8') as f:
    f.write(codigo_micropython.strip())

print('Código MicroPython guardado en firmware/esp32_main.py')
print(f'\nInstrucciones:')
print('  1. Abre Thonny IDE')
print('  2. Conecta el ESP32 por USB')
print('  3. Selecciona intérprete: MicroPython (ESP32)')
print('  4. Abre firmware/esp32_main.py')
print('  5. Edita WIFI_SSID, WIFI_PASSWORD y API_HOST')
print('  6. Guarda como main.py en el ESP32 (File → Save as → MicroPython device)')

---
## 7.7 — Código para Arduino Mega (C++ / Wire HTTP)

Si usas Arduino Mega con shield Ethernet W5100 o módulo ESP8266 como coprocessor.

In [ ]:
codigo_arduino = '''
/*
 * Sistema de Monitoreo de Calidad del Agua
 * Arduino Mega + ESP8266 (AT Commands) o Shield Ethernet W5100
 * Universidad Popular del Cesar — 2026
 */
#include <ArduinoJson.h>   // https://arduinojson.org/
#include <SoftwareSerial.h>

// ─── Pines ───────────────────────────────────────────────────
#define PH_PIN      A0
#define TURB_PIN    A1
#define TEMP_PIN    2    // DS18B20 OneWire
#define LED_OPTIMA  8
#define LED_ALERTA  9
#define LED_CONTAM  10

// ─── API ─────────────────────────────────────────────────────
const char* API_HOST = "192.168.1.X";  // IP del PC
const int   API_PORT = 8000;

SoftwareSerial esp8266(11, 12);  // RX, TX del módulo ESP8266

float leerPH() {
    int raw = 0;
    for (int i = 0; i < 10; i++) raw += analogRead(PH_PIN);
    float voltaje = (raw / 10.0) * 5.0 / 1023.0;
    return constrain(3.5 * voltaje, 0.0, 14.0);
}

float leerTurbidez() {
    int raw = 0;
    for (int i = 0; i < 10; i++) raw += analogRead(TURB_PIN);
    float voltaje = (raw / 10.0) * 5.0 / 1023.0;
    float ntu = voltaje < 2.5 ? 3000.0 :
                -1120.4 * voltaje * voltaje + 5742.3 * voltaje - 4352.9;
    return max(0.0f, ntu);
}

bool enviarAPI(float ph, float turbidez, float temperatura) {
    // Construir JSON
    StaticJsonDocument<200> doc;
    doc["ph"]          = ph;
    doc["turbidez"]    = turbidez;
    doc["temperatura"] = temperatura;
    doc["punto_monitoreo"] = "Guatapuri-Arduino";

    char body[200];
    serializeJson(doc, body);
    int bodyLen = strlen(body);

    // Enviar via ESP8266 AT commands (simplificado)
    esp8266.println("AT+CIPSTART=\"TCP\",\"" + String(API_HOST) + "\"," + API_PORT);
    delay(1000);

    String request = "POST /predict/?modelo=random_forest HTTP/1.1\\r\\n";
    request += "Host: " + String(API_HOST) + "\\r\\n";
    request += "Content-Type: application/json\\r\\n";
    request += "Content-Length: " + String(bodyLen) + "\\r\\n\\r\\n";
    request += String(body);

    esp8266.println("AT+CIPSEND=" + String(request.length()));
    delay(500);
    esp8266.print(request);
    return true;
}

void activarLEDs(String estado) {
    digitalWrite(LED_OPTIMA, estado == "Optima"      ? HIGH : LOW);
    digitalWrite(LED_ALERTA, estado == "Alerta"      ? HIGH : LOW);
    digitalWrite(LED_CONTAM, estado == "Contaminada" ? HIGH : LOW);
}

void setup() {
    Serial.begin(9600);
    esp8266.begin(9600);
    pinMode(LED_OPTIMA, OUTPUT);
    pinMode(LED_ALERTA, OUTPUT);
    pinMode(LED_CONTAM, OUTPUT);
    Serial.println("Sistema iniciado.");
}

void loop() {
    float ph    = leerPH();
    float turb  = leerTurbidez();
    float temp  = 25.0;  // Reemplazar con lectura real DS18B20

    Serial.print("pH=");  Serial.print(ph);
    Serial.print(" Turb="); Serial.print(turb);
    Serial.print(" Temp="); Serial.println(temp);

    if (enviarAPI(ph, turb, temp)) {
        // Leer respuesta del ESP8266 y parsear estado
        // (implementación completa en firmware/arduino_main.ino)
        Serial.println("Datos enviados a la API.");
    }

    delay(300000);  // Esperar 5 minutos
}
'''

with open('../firmware/arduino_main.ino', 'w', encoding='utf-8') as f:
    f.write(codigo_arduino.strip())

print('Código Arduino guardado en firmware/arduino_main.ino')

---
## 7.8 — curl y herramientas externas

Puedes probar la API desde la terminal, Postman o cualquier cliente HTTP:

In [ ]:
print('=== Ejemplos de uso externo de la API ===')
print()
print('── curl (terminal/cmd) ──────────────────────────────────────')
print('''curl -X POST "http://localhost:8000/predict/?modelo=random_forest" \\n     -H "Content-Type: application/json" \\n     -d \'{\'ph\': 7.2, \'turbidez\': 3.5, \'temperatura\': 22.0}\' ''')
print()
print('── Python requests (script externo) ────────────────────────')
print('''
import requests
r = requests.post(
    "http://localhost:8000/predict/?modelo=random_forest",
    json={"ph": 7.2, "turbidez": 3.5, "temperatura": 22.0}
)
print(r.json())
''')
print()
print('── Documentación interactiva ────────────────────────────────')
print('  http://localhost:8000/docs   (Swagger UI)')
print('  http://localhost:8000/redoc  (ReDoc)')
print()
print('── Health check ─────────────────────────────────────────────')
resp = httpx.get(f'{API_BASE}/monitor/health', timeout=5)
print(json.dumps(resp.json(), indent=2, ensure_ascii=False))